# Audio Fundamentals — Waveforms, Sampling, Fourier Transform Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: read a clip and plot the waveform

`code/main.py` uses only the stdlib `wave` module to keep the demo dependency-free. For production you will use `soundfile` or `torchaudio.load` (both return `(waveform, sr)` tuples):

In [ ]:
```python

import soundfile as sf

waveform, sr = sf.read("clip.wav", dtype="float32")  # shape (T,), sr=int

In [ ]:
```

### Step 2: synthesize a sine wave from first principles

In [ ]:
```python

import math

def sine(freq_hz, sr, seconds, amp=0.5):

    n = int(sr * seconds)

    return [amp * math.sin(2 * math.pi * freq_hz * i / sr) for i in range(n)]

In [ ]:
```

A 440 Hz sine (concert A) at 16 kHz for 1 second is 16,000 floats. Write with `wave.open(..., "wb")` using 16-bit PCM encoding.

### Step 3: compute the DFT by hand

In [ ]:
```python

def dft(x):

    N = len(x)

    out = []

    for k in range(N):

        re = sum(x[n] * math.cos(-2 * math.pi * k * n / N) for n in range(N))

        im = sum(x[n] * math.sin(-2 * math.pi * k * n / N) for n in range(N))

        out.append((re, im))

    return out

In [ ]:
```

`O(N²)` — fine for `N=256` to confirm correctness, useless for real audio. Real code calls `numpy.fft.rfft` or `torch.fft.rfft`.

### Step 4: find the dominant frequency

Magnitude peak index `k_star` maps to frequency `k_star * sr / N`. Running this on the 440 Hz sine should return a peak at bin `440 * N / sr`.

### Step 5: demonstrate aliasing

Sample a 7 kHz sine at 10 kHz (Nyquist = 5 kHz). The 7 kHz tone is above Nyquist and folds to `10 − 7 = 3 kHz`. The FFT peak appears at 3 kHz. This is the classic aliasing demo and the reason every DAC/ADC ships with a brick-wall low-pass filter.

## Exercises

In [ ]:
1. **Easy.** Synthesize a 1-second mix of 220 Hz + 440 Hz + 880 Hz at 16 kHz. Run DFT. Confirm three peaks at the expected bins.
2. **Medium.** Record a 3-second WAV of your voice at 48 kHz. Downsample to 16 kHz using `torchaudio.transforms.Resample` (with anti-aliasing), then to 16 kHz using naive decimation (every third sample). FFT both. Where does the aliasing appear?
3. **Hard.** Build the STFT from scratch using only `math` and the DFT from Step 3. Frame size 400, hop 160, Hann window. Plot magnitudes with `matplotlib.pyplot.imshow`. This is the spectrogram of Lesson 02.